## ETL Workflow Automation

### Bronze Zone

In [0]:
spark.conf.set(
  "fs.azure.account.key.<your-storage-account>.dfs.core.windows.net",
  "<ACCESS_KEY>"
)

display(dbutils.fs.ls("abfss://<container>@<storage-account>.dfs.core.windows.net/"))

df_bronze = spark.read.option("header", "true").csv("abfss://<container>@<storage-account>.dfs.core.windows.net/<file-path>")
display(df_bronze.limit(5))




path,name,size,modificationTime
abfss://etl@nevethastoragegen2.dfs.core.windows.net/Netflix Dataset.csv,Netflix Dataset.csv,2962357,1755956565000
abfss://etl@nevethastoragegen2.dfs.core.windows.net/gold/,gold/,0,1755951454000
abfss://etl@nevethastoragegen2.dfs.core.windows.net/silver/,silver/,0,1755951121000


Show_Id,Category,Title,Director,Cast,Country,Release_Date,Rating,Duration,Type,Description
s1,TV Show,3%,null,"João Miguel, Bianca Comparato, Michel Gomes, Rodolfo Valente, Vaneza Oliveira, Rafael Lozano, Viviane Porto, Mel Fronckowiak, Sergio Mamberti, Zezé Motta, Celso Frateschi",Brazil,"August 14, 2020",TV-MA,4 Seasons,"International TV Shows, TV Dramas, TV Sci-Fi & Fantasy","In a future where the elite inhabit an island paradise far from the crowded slums, you get one chance to join the 3% saved from squalor."
s2,Movie,07:19,Jorge Michel Grau,"Demián Bichir, Héctor Bonilla, Oscar Serrano, Azalia Ortiz, Octavio Michel, Carmen Beato",Mexico,"December 23, 2016",TV-MA,93 min,"Dramas, International Movies","After a devastating earthquake hits Mexico City, trapped survivors from all walks of life wait to be rescued while trying desperately to stay alive."
s3,Movie,23:59,Gilbert Chan,"Tedd Chan, Stella Chung, Henley Hii, Lawrence Koh, Tommy Kuan, Josh Lai, Mark Lee, Susan Leong, Benjamin Lim",Singapore,"December 20, 2018",R,78 min,"Horror Movies, International Movies","When an army recruit is found dead, his fellow soldiers are forced to confront a terrifying secret that's haunting their jungle island training camp."
s4,Movie,9,Shane Acker,"Elijah Wood, John C. Reilly, Jennifer Connelly, Christopher Plummer, Crispin Glover, Martin Landau, Fred Tatasciore, Alan Oppenheimer, Tom Kane",United States,"November 16, 2017",PG-13,80 min,"Action & Adventure, Independent Movies, Sci-Fi & Fantasy","In a postapocalyptic world, rag-doll robots hide in fear from dangerous machines out to exterminate them, until a brave newcomer joins the group."
s5,Movie,21,Robert Luketic,"Jim Sturgess, Kevin Spacey, Kate Bosworth, Aaron Yoo, Liza Lapira, Jacob Pitts, Laurence Fishburne, Jack McGee, Josh Gad, Sam Golzari, Helen Carey, Jack Gilpin",United States,"January 1, 2020",PG-13,123 min,Dramas,A brilliant group of students become card-counting experts with the intent of swindling millions out of Las Vegas casinos by playing blackjack.


### Silver Zone

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType

schema = StructType([
    StructField("Show_Id", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Title", StringType(), True),
    StructField("Director", StringType(), True),
    StructField("Cast", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("Release_Date", StringType(), True),  # parse later
    StructField("Rating", StringType(), True),
    StructField("Duration", StringType(), True),
    StructField("Type", StringType(), True),
    StructField("Description", StringType(), True)
])

df_bronze = spark.read.option("header", "true") \
    .schema(schema) \
    .csv("abfss://<container>@<storage-account>.dfs.core.windows.net/<file-path>")

from pyspark.sql.functions import col, to_date, trim

df_silver = df_bronze.dropDuplicates(["Show_Id"]) \
    .na.drop(subset=["Title", "Category"]) \
    .withColumn("Release_Date", to_date(col("Release_Date"), "MMMM d, yyyy"))

# Trim all string columns
for c in df_silver.columns:
    df_silver = df_silver.withColumn(c, trim(col(c)))

df_silver.write.mode("overwrite").parquet(
    "abfss://<container>@<storage-account>.dfs.core.windows.net/<file-path>")

df_check = spark.read.parquet("abfss://<container>@<storage-account>.dfs.core.windows.net/<file-path>")
display(df_check.limit(10))





Show_Id,Category,Title,Director,Cast,Country,Release_Date,Rating,Duration,Type,Description
s82,Movie,2015 Dream Concert,null,"4Minute, B1A4, BtoB, ELSIE, EXID, EXO, Got7, INFINITE, KARA, Shinee, Sistar, VIXX, Nine Muses, BTS, Secret, Topp Dogg",South Korea,2017-04-28,TV-PG,107 min,"International Movies, Music & Musicals","The world's biggest K-pop festival marked its 21st year in 2015, with groups such as EXO, 4Minute and SHINee electrifying the Seoul World Cup Stadium."
s432,Movie,Along Came a Spider,Lee Tamahori,"Morgan Freeman, Monica Potter, Michael Wincott, Dylan Baker, Mika Boorem, Anton Yelchin, Kim Hawthorne, Jay O. Sanders, Billy Burke, Michael Moriarty, Penelope Ann Miller","United States, Germany, Canada",2019-10-01,R,103 min,Thrillers,"When a girl is kidnapped from a prestigious prep school, a homicide detective takes the case, teaming up with young security agent."
s559,TV Show,Apache: The Life of Carlos Tevez,null,"Balthazar Murillo, Vanesa González, Alberto Ajaka, Sofía Gala Castiglione, Patricio Contreras, Matías Recalt, Osqui Guzmán, Roberto Vallejos, Diego Gallardo",Argentina,2019-08-16,TV-MA,1 Season,"Crime TV Shows, International TV Shows, Spanish-Language TV Shows",This gritty dramatization of the life of Carlos Tevez shows his rise to soccer stardom amid the harrowing conditions in Argentina's Fuerte Apache.
s853,Movie,Best of Stand-Up 2020,null,"Jerry Seinfeld, Leslie Jones, Taylor Tomlinson, Tom Segura, Jack Whitehall, Michelle Buteau, Bert Kreischer, Jo Koy, Donnell Rawlings, Jim Jefferies, Nikki Glaser, George Lopez, Sam Jay, Marc Maron, Kevin Hart, Michael McIntyre, Fortune Feimster, Eric André, Jim Norton, Felipe Esparza, Hannah Gadsby, Patton Oswalt, Vir Das, Robert Kelly, Urzila Carlson, Tom Papa, Kanan Gill, Ms. Pat, Rob Schneider, Adrienne Iapalucci, Kenny Sebastian, Thomas Middleditch, Ben Schwartz",null,2020-12-31,TV-MA,77 min,Stand-Up Comedy,"From Jerry Seinfeld to Leslie Jones, Kevin Hart to Hannah Gadsby, laugh along with the funniest bits from Netflix's 2020 stand-up comedy specials."
s971,Movie,BLAME!,Hiroyuki Seshita,"Takahiro Sakurai, Kana Hanazawa, Sora Amamiya, Kazuhiro Yamaji, Mamoru Miyano, Aya Suzaki, Nobunaga Shimazaki, Yuki Kaji, Aki Toyosaki, Saori Hayami",Japan,2017-05-19,TV-14,106 min,"Action & Adventure, Anime Features, International Movies","Inside a vast, self-replicating city bent on eliminating all life, mysterious loner Killy emerges to guide a remnant of humanity desperate to survive."
s974,TV Show,Bleach,null,"Masakazu Morita, Fumiko Orikasa, Yuki Matsuoka, Noriaki Sugiyama, Hiroki Yasumoto, Kentaro Ito, Ryotaro Okiayu",Japan,2020-04-21,TV-14,5 Seasons,"Anime Series, International TV Shows","After teenager Ichigo Kurosaki acquires superpowers from wounded soul reaper Rukia Kuchiki, the two of them join forces to round up lost souls."
s1102,Movie,Bridget Christie: Stand Up for Her,null,Bridget Christie,United Kingdom,2017-03-31,TV-MA,51 min,Stand-Up Comedy,"Performing stand-up for a packed house in London's Hoxton Hall, comedian Bridget Christie dives into the politics of gender, sex and equality."
s1196,Movie,Can't Help Falling in Love,Mae Czarina Cruz,"Kathryn Bernardo, Daniel Padilla, Matteo Guidicelli, Zanjoe Marudo, Cherry Pie Picache, Lotlot De Leon, Dennis Padilla, Lito Pimentel, Joross Gamboa, Janus del Prado",Philippines,2019-02-27,TV-14,120 min,"Comedies, Dramas, International Movies","Gab is eager to tie the knot with her handsome boyfriend, but there's a problem, and it's a doozy: She's already married to a total stranger."
s1362,Movie,Christine,Antonio Campos,"Rebecca Hall, Michael C. Hall, Tracy Letts, Maria Dizzia, J. Smith-Cameron, Timothy Simons, Kim Shaw, John Cullum, Morgan Spector, Jayson Warner Smith","United Kingdom, United States",2020-08-13,R,119 min,"Dramas, Independent Movies","In a film based on true events, an awkward but ambitious TV reporter struggles to adapt when she's ordered to focus on violent and salacious stories."
s1453,Movie,Coffee

### Gold Zone

In [0]:
df_silver = spark.read.parquet(
    "abfss://<container>@<storage-account>.dfs.core.windows.net/<file-path>")

from pyspark.sql.functions import count

gold_category = df_silver.groupBy("Category").agg(count("*").alias("Total_Shows"))
display(gold_category)

from pyspark.sql.functions import year

gold_year = df_silver.withColumn("Year", year("Release_Date")) \
    .groupBy("Year").agg(count("*").alias("Total_Releases")) \
    .orderBy("Year")
display(gold_year)

gold_country = df_silver.groupBy("Country") \
    .agg(count("*").alias("Total_Titles")) \
    .orderBy(col("Total_Titles").desc()) \
    .limit(10)

display(gold_country)

gold_rating = df_silver.groupBy("Rating") \
    .agg(count("*").alias("Total_Titles")) \
    .orderBy(col("Total_Titles").desc())

display(gold_rating)

gold_category.write.mode("overwrite").parquet(
    "abfss://<container>@<storage-account>.dfs.core.windows.net/<file-path>")

gold_year.write.mode("overwrite").parquet(
    "abfss://<container>@<storage-account>.dfs.core.windows.net/<file-path>")

gold_country.write.mode("overwrite").parquet(
    "abfss://<container>@<storage-account>.dfs.core.windows.net/<file-path>")

gold_rating.write.mode("overwrite").parquet(
    "abfss://<container>@<storage-account>.dfs.core.windows.net/<file-path>")

df_gold_check = spark.read.parquet(
    "abfss://<container>@<storage-account>.dfs.core.windows.net/<file-path>")
display(df_gold_check)



Category,Total_Shows
TV Show,2410
Movie,5377


Year,Total_Releases
null,113
2008,2
2009,2
2010,1
2011,13
2012,3
2013,10
2014,24
2015,78
2016,432


Country,Total_Titles
United States,2543
India,923
null,508
United Kingdom,397
Japan,226
South Korea,183
Canada,177
Spain,134
France,114
Egypt,101


Rating,Total_Titles
TV-MA,2854
TV-14,1929
TV-PG,805
R,663
PG-13,386
TV-Y,280
TV-Y7,271
PG,246
TV-G,194
NR,84


Year,Total_Releases
null,113
2008,2
2009,2
2010,1
2011,13
2012,3
2013,10
2014,24
2015,78
2016,432
